# Phase 5: Persistent Embedding Cache

The Twelve Labs `embed.v_2` call is the slowest part of the pipeline — every video pays an upload plus an inference cost. This stage implements a local cache so the same file reappearing in a later run (or even a different FiftyOne dataset) skips the API entirely.

**Design at a glance:**
- **Key:** SHA-256 of the first 10 MB of each file — enough prefix to tell videos apart while staying cheap on large files.
- **Value:** the 512-d Marengo embedding serialized as `float32` bytes, alongside dim + dtype for safe deserialization.
- **Store:** SQLite at `~/.video_gap_analyzer/embeddings.db` (created on first use).
- **Lifetime:** survives across processes, datasets, and re-creations of a dataset pointing at the same files.

This notebook contains the full implementation of `EmbeddingCache` inline so it stays self-contained like the other stage notebooks. The plugin (`__init__.py`) imports the same class from `embedding_cache.py` at the repo root.

## 1. Configuration

A handful of tunables: how much of each file we hash, where the database lives, and what numeric type we serialize embeddings as.

In [1]:
from __future__ import annotations

import hashlib
import logging
import os
import sqlite3
import time
from pathlib import Path
from typing import Optional, Union

import numpy as np

logger = logging.getLogger(__name__)

# First 10 MB is enough to tell videos apart while staying cheap on large
# files. Reading less risks collisions on shared intros (logos, bumpers).
HASH_BYTES = 10 * 1024 * 1024

DEFAULT_CACHE_DIR = Path.home() / ".video_gap_analyzer"
DEFAULT_CACHE_DB = DEFAULT_CACHE_DIR / "embeddings.db"

# Marengo embeddings arrive as JSON-serialized floats; float32 is lossless
# relative to the payload and halves the on-disk footprint vs float64.
DEFAULT_DTYPE = "float32"

## 2. SQLite schema

One table keyed by content hash. Storing `dim` and `dtype` alongside the raw bytes means we can safely deserialize even if the embedding model changes in the future.

In [2]:
_SCHEMA = """
CREATE TABLE IF NOT EXISTS embeddings (
    file_hash  TEXT PRIMARY KEY,
    embedding  BLOB NOT NULL,
    dim        INTEGER NOT NULL,
    dtype      TEXT NOT NULL,
    created_at REAL NOT NULL
);
"""

## 3. Content-based hashing

The cache identifies videos by their content, not their filename. Two datasets referencing the same underlying file get the same hash — and therefore hit the same cache entry. Hashing only the first 10 MB keeps this near-instant even for multi-GB files.

In [3]:
def compute_video_hash(filepath: Union[str, os.PathLike], num_bytes: int = HASH_BYTES) -> str:
    """Return the SHA-256 hex digest of the first ``num_bytes`` of ``filepath``."""
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        h.update(f.read(num_bytes))
    return h.hexdigest()

## 4. The `EmbeddingCache` class

Five methods make up the public API:

| Method | What it does |
|--------|--------------|
| `get(filepath)` | Hash the file, look it up, return a `float32` array or `None` on miss |
| `put(filepath, embedding)` | Hash the file, `INSERT OR REPLACE` the embedding |
| `clear()` | `DELETE FROM embeddings`, returns number of rows wiped |
| `stats()` | `{entries, db_path, size_bytes}` for introspection |
| `close()` | Release the SQLite connection |

It also supports `with` for automatic cleanup.

In [4]:
class EmbeddingCache:
    """SQLite-backed cache mapping file-content hashes to embedding vectors."""

    def __init__(self, db_path: Optional[Union[str, os.PathLike]] = None):
        self.db_path = Path(db_path) if db_path else DEFAULT_CACHE_DB
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        self.conn = sqlite3.connect(str(self.db_path))
        self.conn.executescript(_SCHEMA)
        self.conn.commit()

    def get(self, filepath: Union[str, os.PathLike]) -> Optional[np.ndarray]:
        """Return the cached embedding for ``filepath``'s content, or ``None``."""
        try:
            file_hash = compute_video_hash(filepath)
        except OSError as e:
            logger.warning("Could not hash %s: %s", filepath, e)
            return None

        row = self.conn.execute(
            "SELECT embedding, dim, dtype FROM embeddings WHERE file_hash = ?",
            (file_hash,),
        ).fetchone()
        if row is None:
            return None

        blob, dim, dtype = row
        arr = np.frombuffer(blob, dtype=np.dtype(dtype))
        if arr.size != dim:
            logger.warning(
                "Cache entry %s is corrupted (size %d != dim %d); ignoring",
                file_hash[:12], arr.size, dim,
            )
            return None
        return arr.copy()

    def put(self, filepath: Union[str, os.PathLike], embedding) -> str:
        """Store ``embedding`` under ``filepath``'s content hash. Returns the hash."""
        file_hash = compute_video_hash(filepath)
        arr = np.asarray(embedding, dtype=np.dtype(DEFAULT_DTYPE)).reshape(-1)
        self.conn.execute(
            "INSERT OR REPLACE INTO embeddings "
            "(file_hash, embedding, dim, dtype, created_at) VALUES (?, ?, ?, ?, ?)",
            (file_hash, arr.tobytes(), int(arr.size), DEFAULT_DTYPE, time.time()),
        )
        self.conn.commit()
        return file_hash

    def clear(self) -> int:
        """Delete all cache entries. Returns the number of rows removed."""
        cur = self.conn.execute("DELETE FROM embeddings")
        self.conn.commit()
        return int(cur.rowcount)

    def stats(self) -> dict:
        """Return {entries, db_path, size_bytes} describing the cache on disk."""
        count = self.conn.execute("SELECT COUNT(*) FROM embeddings").fetchone()[0]
        size_bytes = self.db_path.stat().st_size if self.db_path.exists() else 0
        return {
            "entries": int(count),
            "db_path": str(self.db_path),
            "size_bytes": int(size_bytes),
        }

    def close(self) -> None:
        self.conn.close()

    def __enter__(self) -> "EmbeddingCache":
        return self

    def __exit__(self, *exc) -> None:
        self.close()

## 5. Demo setup

Pick any video file on disk — a FiftyOne quickstart video works well. Only the first 10 MB get read, so file size doesn't matter.

In [5]:
import glob

# Pick any video file under ~/fiftyone — the quickstart-video dataset drops
# ~50 short clips here. Only the first 10 MB get read, so file size doesn't
# matter. If you already have a specific file in mind, set VIDEO_PATH directly.
candidates = sorted(glob.glob(str(Path("~/fiftyone/quickstart-video/data").expanduser() / "*.mp4")))
if not candidates:
    candidates = sorted(glob.glob(str(Path("~/fiftyone").expanduser() / "**" / "*.mp4"), recursive=True))
assert candidates, "No .mp4 files found under ~/fiftyone — set VIDEO_PATH manually"

VIDEO_PATH = Path(candidates[0])
print(f"Using: {VIDEO_PATH}")
print(f"Size:  {VIDEO_PATH.stat().st_size / 1e6:.1f} MB")

Using: /Users/surabhigade/fiftyone/quickstart-video/data/0587e1cfc2344523922652d8b227fba4-000014-video_052.mp4
Size:  9.9 MB


## 6. Hashing the file

The hash is stable as long as the first 10 MB don't change. Re-encoding a video or trimming its header will change the hash (and force a fresh embed) — that's the right behavior, since the new bytes are a different video as far as Marengo is concerned.

In [6]:
file_hash = compute_video_hash(VIDEO_PATH)
print(f"file_hash = {file_hash}")

file_hash = 2f4b59b376c076244816456e5a218710513cad8a5a3a8467333ebdf4b7d786bd


## 7. Round-trip: put → get

In production, `embed_sample` calls `cache.put(filepath, response.data[0].embedding)` immediately after a successful Marengo call, then every subsequent run hits `cache.get(filepath)` first and short-circuits the API when the hash matches.

In [7]:
with EmbeddingCache() as cache:
    print(f"cache db: {cache.db_path}")

    fake = np.random.default_rng(0).standard_normal(512).astype("float32")
    cache.put(VIDEO_PATH, fake)

    retrieved = cache.get(VIDEO_PATH)
    assert retrieved is not None and np.allclose(retrieved, fake)
    print(f"round trip ok — shape={retrieved.shape} dtype={retrieved.dtype}")
    print(f"stats: {cache.stats()}")

cache db: /Users/surabhigade/.video_gap_analyzer/embeddings.db
round trip ok — shape=(512,) dtype=float32
stats: {'entries': 1, 'db_path': '/Users/surabhigade/.video_gap_analyzer/embeddings.db', 'size_bytes': 12288}


## 8. Cache miss

Any unseen path — or a path that errors on read — returns `None` without raising, which the pipeline treats as "fall through to the Twelve Labs API".

In [8]:
with EmbeddingCache() as cache:
    miss = cache.get("/tmp/this-file-does-not-exist.mp4")
    print(f"miss returned: {miss!r}")

Could not hash /tmp/this-file-does-not-exist.mp4: [Errno 2] No such file or directory: '/tmp/this-file-does-not-exist.mp4'


miss returned: None


## 9. Clearing the cache

`EmbeddingCache.clear()` wipes every row and returns the count. The `AnalyzeCoverage` operator exposes this via its **Clear Embedding Cache** checkbox — when checked, the operator calls `cache.clear()` and also nulls every sample's `embedding` field, so stage 1 hits Marengo fresh for every video.

In [9]:
with EmbeddingCache() as cache:
    before = cache.stats()
    deleted = cache.clear()
    after = cache.stats()
    print(f"before: {before['entries']} entries")
    print(f"deleted: {deleted}")
    print(f"after:  {after['entries']} entries")

before: 1 entries
deleted: 1
after:  0 entries


## 10. How the plugin uses it

Inside `__init__.py`, the embed stage opens a cache once and passes it to every concurrent task:

```python
async with AsyncTwelveLabs(api_key=api_key) as client_async:
    with EmbeddingCache() as cache:
        tasks = [
            asyncio.create_task(_embed_one_async(client_async, s, cache, semaphore))
            for s in samples
        ]
        for coro in asyncio.as_completed(tasks):
            ...
```

Each `_embed_one_async` checks `cache.get(filepath)` *before* acquiring the semaphore — so cache hits don't consume a concurrency slot. After a successful Marengo call it calls `cache.put(filepath, embedding)`. The result: a re-run on the same dataset finishes in under two seconds regardless of how many videos it contains.